In [ ]:
# ===================================================================================
# APP MIGRATION PRE-FLIGHT CHECK (V2.0 - Hardened)
# - UPDATED: Added Session/Retry logic to prevent False Negatives on searches.
# - Scans Dashboards & Web Experiences for Web Map dependencies.
# - Checks if those Web Maps already exist on Target.
# - OUTPUT: List of Missing Web Map IDs to copy (feeds your Web Map Migrator).
# ===================================================================================

import pandas as pd
import json
import os
import arcgis
import urllib3
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
# LIST THE APPS (DASHBOARDS OR EXPERIENCES) TO CHECK

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_preflight_apps.json")
if os.path.exists(_sidecar_path):
    import json as _json
    APP_IDS_TO_CHECK = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(APP_IDS_TO_CHECK)} App IDs from sidecar.")
else:
    APP_IDS_TO_CHECK = [
        # Example Source ID's
        # "7b295fd36da448c1a4ced27681da3aed",
        # "1132c531509e4000a293fef3d39db052", 
        # "7f7823f9d0ee4ad6b6c885e75d9fa76f",
    ]

# --- LOAD HISTORY ---
ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if 'SourceID' in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log['SourceID'].astype(str).str.strip())
    except: pass

# --- CONNECT (HARDENED) ---
print("Connecting for App Scan...")
try:
    session = requests.Session()
    retry_strategy = Retry(
        total=5, 
        backoff_factor=1, 
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    exit()

# --- HELPER: PARSE APP DEPENDENCIES ---
def get_app_dependencies(item):
    """Parses Dashboard or Experience JSON to find Web Map IDs."""
    dependencies = set()
    
    try:
        data = item.get_data()
        if not data: return dependencies

        # A. If it's a DASHBOARD
        if item.type == "Dashboard":
            widgets = data.get('widgets', [])
            for w in widgets:
                if w.get('type') == 'mapWidget':
                    if 'itemId' in w:
                        dependencies.add(w['itemId'])
        
        # B. If it's a WEB EXPERIENCE
        elif item.type == "Web Experience":
            data_sources = data.get('dataSources', {})
            for ds_id, ds_info in data_sources.items():
                if 'itemId' in ds_info:
                    dependencies.add(ds_info['itemId'])
                    
    except Exception as e:
        print(f"   ⚠️ Error parsing JSON for {item.title}: {e}")
        
    return dependencies

# --- MAIN EXECUTION ---
print(f"\nScanning {len(APP_IDS_TO_CHECK)} Apps for Web Map dependencies...\n")

total_missing_maps = set()
apps_ready = []
apps_blocked = []

for app_id in APP_IDS_TO_CHECK:
    try:
        app_item = source_gis.content.get(app_id)
        if not app_item:
            print(f"❌ App {app_id} not found.")
            continue
            
        # Get list of Web Maps this App uses
        required_maps = get_app_dependencies(app_item)
        
        missing_maps = set()
        found_maps = set()
        
        for map_id in required_maps:
            # CHECK 1: Is it in CSV?
            if str(map_id) in ALREADY_MIGRATED_IDS:
                found_maps.add(map_id)
                continue

            # CHECK 2: Is it on Target Portal? (Live Check via Tags)
            try:
                search_query = f'tags:"SourceID:{map_id}"'
                res = target_gis.content.search(search_query, max_items=1)
                
                if res:
                    found_maps.add(map_id)
                else:
                    missing_maps.add(map_id)
            except Exception as search_err:
                print(f"   ⚠ Search Error for map {map_id}: {search_err}")
                # Assume missing if search fails hard
                missing_maps.add(map_id)

        # REPORT STATUS
        if missing_maps:
            print(f"🔴 BLOCKED: {app_item.title} ({app_item.type})")
            print(f"   Missing {len(missing_maps)} Web Maps: {list(missing_maps)}")
            apps_blocked.append(app_item.title)
            total_missing_maps.update(missing_maps)
        else:
            print(f"🟢 READY:   {app_item.title} ({app_item.type})")
            apps_ready.append(app_item.title)
            
    except Exception as e:
        print(f"❌ Error scanning {app_id}: {e}")

# --- FINAL REPORT ---
print("\n" + "="*50)
print("              APP PRE-FLIGHT REPORT")
print("="*50)
print(f"✅ Apps Ready to Migrate:       {len(apps_ready)}")
print(f"🔴 Apps Missing Web Maps:       {len(apps_blocked)}")
print("-" * 50)

if total_missing_maps:
    print("\n⚠️  MISSING WEB MAPS DETECTED")
    print("Copy the list below into your Web Map Migration Script:\n")
    
    print("WEB_MAP_IDS_TO_MIGRATE = [")
    for mid in total_missing_maps:
        try:
            item = source_gis.content.get(mid)
            name = item.title if item else "Unknown"
            print(f'    "{mid}", # {name}')
        except:
            print(f'    "{mid}",')
    print("]")
else:
    print("\n🚀 All Apps are ready! Their Web Maps exist on Target.")
    print("You can now run your Dashboard/Experience Migration Script.")

print("="*50)

# --- ORCHESTRATOR OUTPUT (write missing IDs for downstream consumption) ---
import json as _json
_output_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_output_preflight_apps.json")
with open(_output_path, "w", encoding="utf-8") as _f:
    _json.dump({"missing_ids": list(total_missing_maps)}, _f, indent=2)
print(f"[Orchestrator] Wrote {len(total_missing_maps)} missing Web Map IDs to {os.path.basename(_output_path)}")